# 01 - Data Pipeline

## MLB Ballpark Effects on Pitch Type Frequency & Contact Quality

This notebook functions as a **one-time data pipeline** for the full project.  It pulls Statcast pitch-level data from the pybaseball library package, filtering to swing events only, samples 2,000 swings per park per season, saving the final 174,000 row dataset for any further analysis.

*Seasons*: 2023, 2024, 2025
<br>
*Parks*: 29 Major League Ballparks (Athletics excluded)
<br>
*Total Sample*: 174,000 swings (2,000 swings per season x 3 seasons x 29 parks)

### Imports:

In [8]:
import pandas as np
import numpy as np
import pybaseball
from pybaseball import statcast
import time
import os
import warnings

warnings.filterwarnings('ignore')

pybaseball.cache.enable()

print("Imports Successful")

Imports Successful


### Constants and Configuration:

In [ ]:
# Seasons
Seasons = [2023, 2024, 2025]


# Monthly Date Ranges per Season
Season_Months = {
    2023: [
        ('2023-03-30', '2023-04-30'),
        ('2023-05-01', '2023-05-31'),
        ('2023-06-01', '2023-06-30'),
        ('2023-07-01', '2023-07-31'),
        ('2023-08-01', '2023-08-31'),
        ('2023-09-01', '2023-10-01'),
    ],
    2024: [
        ('2024-03-20', '2024-04-30'),
        ('2024-05-01', '2024-05-31'),
        ('2024-06-01', '2024-06-30'),
        ('2024-07-01', '2024-07-31'),
        ('2024-08-01', '2024-08-31'),
        ('2024-09-01', '2024-09-30'),
    ],
    2025: [
        ('2025-03-27', '2025-04-30'),
        ('2025-05-01', '2025-05-31'),
        ('2025-06-01', '2025-06-30'),
        ('2025-07-01', '2025-07-31'),
        ('2025-08-01', '2025-08-31'),
        ('2025-09-01', '2025-09-28'),
    ],
}


# 29 MLB Parks (home_team abbreviations)
Ballparks = [
    'AZ',  'ATL', 'BAL', 'BOS', 'CHC',
    'CWS', 'CIN', 'CLE', 'COL', 'DET',
    'HOU', 'KC',  'LAA', 'LAD', 'MIA',
    'MIL', 'MIN', 'NYM', 'NYY', 'PHI',
    'PIT', 'SD',  'SF',  'SEA', 'STL',
    'TB',  'TEX', 'TOR', 'WSH'
]


# Swing Descriptions
Swing_Descriptions = [
    'swinging_strike',
    'swinging_strike_blocked',
    'foul',
    'foul_tip',
    'hit_into_play',
    'hit_into_play_no_out',
    'hit_into_play_score',
]


# Keep Columns
Keep_Cols = [
    'game_date',
    'home_team',
    'pitch_type',
    'description',
    'launch_speed',
    'launch_angle',
    'stand',
    'p_throws',
    'release_speed',
]


# Sampling
Swings_Per_Park = 2000
Random_Seed = 2005 # Go White Sox!

# Output Paths
# need
# need

print("Configuration loaded.")
print(f"Parks: {len(Ballparks)}")
print(f"Seasons: {Seasons}")
print(f"Target sample: {len(Ballparks) * Swings_Per_Park * len(Seasons):,} total swings")


Configuration loaded.
Parks: 29
Seasons: [2023, 2024, 2025]
Target sample: 174,000 total swings


### Pitch Categorization and Outcome Classification Functions:

In [10]:
def classify_pitch(pitch_type):
    """Map Statcast pitch type codes to Level 1 categories."""
    fastballs = ['FA', 'FF', 'FT', 'FC', 'SI', 'SF']
    breaking  = ['CU', 'KC', 'SL', 'ST', 'SV']
    offspeed  = ['CH', 'CS', 'EP', 'FO', 'SC']
    
    if pitch_type in fastballs:
        return 'Fastball'
    elif pitch_type in breaking:
        return 'Breaking'
    elif pitch_type in offspeed:
        return 'Offspeed'
    else:
        return 'Other'

def classify_outcome(description):
    """Map Statcast description to swing outcome category."""
    if description in ['swinging_strike', 
                       'swinging_strike_blocked', 
                       'foul_tip']:
        return 'whiff'
    elif description in ['foul']:
        return 'foul'
    elif description in ['hit_into_play', 
                         'hit_into_play_no_out', 
                         'hit_into_play_score']:
        return 'in_play'
    else:
        return 'other'

print("Classification functions defined.")

Classification functions defined.


### Monthly Pull Function:

In [ ]:
def pull_month(start_date, end_date, retries = 3, delay = 10):
    """
    Pull one month of Statcast data w/ retry logic.
    Returns filtered swing-only dataframe with kept columns only.
    """
    for attempt in range(retries):
        try:
            print(f"  Pulling {start_date} → {end_date} ...", end=" ")
            raw = statcast(start_dt = start_date, end_dt = end_date)
            
            if raw is None or len(raw) == 0:
                print("empty response, skipping.")
                return pd.DataFrame()
            
            # Filter to swings only
            swings = raw[raw['description'].isin(Swing_Descriptions)].copy()
            
            # Filter to our 29 parks
            swings = swings[swings['home_team'].isin(Ballparks)]
            
            # Keep only needed columns (handle missing cols gracefully)
            available = [c for c in Keep_Cols if c in swings.columns]
            swings = swings[available]
            
            print(f"{len(swings):,} swings retained.")
            return swings
        
        except Exception as e:
            print(f"attempt {attempt+1} failed: {e}")
            if attempt < retries - 1:
                print(f"  Retrying in {delay}s...")
                time.sleep(delay)
    
    print(f"  All retries failed for {start_date} → {end_date}. Skipping.")
    return pd.DataFrame()

print("Pull function defined.")

Pull function defined.


###

### Season Pipeline Function:

In [16]:
def build_season_sample(season):
    """
    Pull all months for a season, pool swings,
    sample 2,000 per park, return clean dataframe.
    """
    print(f"\n{'='*50}")
    print(f"SEASON {season}")
    print(f"{'='*50}")
    
    months = Season_Months[season]
    season_swings = []
    
    # Pull month by month
    for start, end in months:
        month_df = pull_month(start, end)
        if len(month_df) > 0:
            season_swings.append(month_df)
        time.sleep(3)  # delay between pulls
    
    if not season_swings:
        print(f"No data retrieved for {season}.")
        return pd.DataFrame()
    
    # Pool all months
    full_season = pd.concat(season_swings, ignore_index=True)
    print(f"\nFull season pool: {len(full_season):,} swings across all parks")
    
    # Add derived columns
    full_season['season'] = season
    full_season['pitch_category'] = full_season['pitch_type'].apply(classify_pitch)
    full_season['swing_outcome'] = full_season['description'].apply(classify_outcome)
    
    # Sample 2,000 per park
    sampled_parts = []
    print(f"\nSampling {Swings_Per_Park} swings per park:")
    
    for park in PARKS:
        park_swings = full_season[full_season['home_team'] == park]
        available = len(park_swings)
        
        if available < SWINGS_PER_PARK:
            print(f"  {park}: only {available} swings available — taking all")
            sampled_parts.append(park_swings)
        else:
            sampled = park_swings.sample(
                n=Swings_Per_Park,
                random_state=Random_Seed
            )
            sampled_parts.append(sampled)
            print(f"  {park}: {available:,} available → {Swings_Per_Park} sampled")
    
    season_sample = pd.concat(sampled_parts, ignore_index=True)
    
    # Save season sample
    #out_path = f"../data/sampled/sample_{season}.csv"
    #season_sample.to_csv(out_path, index=False)
    print(f"\nSeason {season} sample saved → {out_path}")
    print(f"Shape: {season_sample.shape}")
    
    return season_sample

print("Season pipeline function defined.")

Season pipeline function defined.


### Running the Pipeline:

In [ ]:
all_seasons = []

for season in Seasons:
    df_season = build_season_sample(season)
    if len(df_season) > 0:
        all_seasons.append(df_season)

print("\nAll seasons complete.")

### Pool and Save Final Dataset:

In [ ]:
# Pool all three seasons
final = pd.concat(all_seasons, ignore_index=True)

# Final column order
final = final[[
    'season',
    'game_date',
    'home_team',
    'pitch_type',
    'pitch_category',
    'description',
    'swing_outcome',
    'launch_speed',
    'launch_angle',
    'stand',
    'p_throws',
    'release_speed',
]]

# Save
#out_path = '../data/final/mlb_pooled_sample.csv'
final.to_csv(out_path, index=False)

print(f"Final dataset saved → {out_path}")
print(f"Shape: {final.shape}")
print(f"\nSwings per season:")
print(final['season'].value_counts().sort_index())
print(f"\nSwings per park (top 10):")
print(final['home_team'].value_counts().head(10))
print(f"\nPitch category distribution:")
print(final['pitch_category'].value_counts())
print(f"\nSwing outcome distribution:")
print(final['swing_outcome'].value_counts())

### Validation Check:

In [ ]:
print("PIPELINE VALIDATION")
print("=" * 40)

expected_rows = len(PARKS) * SWINGS_PER_PARK * len(SEASONS)
actual_rows = len(final)

print(f"Expected rows:  {expected_rows:,}")
print(f"Actual rows:    {actual_rows:,}")
print(f"Parks present:  {final['home_team'].nunique()} / {len(PARKS)}")
print(f"Seasons present: {sorted(final['season'].unique())}")
print(f"Null launch_speed: {final['launch_speed'].isna().sum():,}")
print(f"Null launch_angle: {final['launch_angle'].isna().sum():,}")
print(f"Null pitch_type:   {final['pitch_type'].isna().sum():,}")

if actual_rows >= expected_rows * 0.95:
    print("\nPIPELINE PASSED — ready for analysis.")
else:
    print("\nWARNING — row count below 95% of expected. Check sampled/ CSVs.")